# Post-Wildfire Burn Severity Exploration

This notebook demonstrates the P1 burn severity analysis pipeline:
1. Load pre/post-fire imagery (synthetic data)
2. Compute NBR and dNBR
3. Classify burn severity
4. Visualize recovery curves
5. Summary statistics

**CRS:** EPSG:3310 (California Albers NAD 83)  
**Sensor:** Sentinel-2 L2A (B8A NIR, B12 SWIR)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

# Ensure project root is on path
project_root = Path.cwd().parents[2] if "notebooks" in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from shared.utils.config import load_config
from shared.utils.io import make_profile, write_raster, read_raster
from shared.data.generate_synthetic import generate_synthetic_burn_rasters
from projects.p1_burn_severity.src.severity import (
    compute_nbr, compute_dnbr, classify_severity, compute_rbr,
)
from projects.p1_burn_severity.src.recovery import (
    compute_vegetation_index, build_recovery_timeseries, fit_recovery_model,
)

print("Imports OK")

## 1. Load Configuration and Synthetic Imagery

In [ ]:
# Load project config
config_path = project_root / "projects" / "p1_burn_severity" / "configs" / "default.yaml"
config = load_config(config_path)
print("Severity thresholds:")
for name, bounds in config["processing"]["severity_thresholds"].items():
    print(f"  {name}: {bounds}")

# Generate synthetic pre/post imagery
output_dir = project_root / "data" / "tmp_notebook"
synth = generate_synthetic_burn_rasters(output_dir)
print(f"\nSynthetic rasters: {list(synth.keys())}")

# Load rasters
pre_nir, profile = read_raster(synth["pre_nir"])
pre_swir, _ = read_raster(synth["pre_swir"])
post_nir, _ = read_raster(synth["post_nir"])
post_swir, _ = read_raster(synth["post_swir"])

# Squeeze to 2-D
pre_nir, pre_swir = pre_nir[0], pre_swir[0]
post_nir, post_swir = post_nir[0], post_swir[0]
print(f"Raster shape: {pre_nir.shape}")

## 2. Compute NBR and dNBR

In [ ]:
# Compute spectral indices
pre_nbr = compute_nbr(pre_nir, pre_swir)
post_nbr = compute_nbr(post_nir, post_swir)
dnbr = compute_dnbr(pre_nbr, post_nbr)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(pre_nbr, cmap="RdYlGn", vmin=-1, vmax=1)
axes[0].set_title("Pre-fire NBR")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(post_nbr, cmap="RdYlGn", vmin=-1, vmax=1)
axes[1].set_title("Post-fire NBR")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(dnbr, cmap="RdYlGn_r", vmin=-0.5, vmax=1.3)
axes[2].set_title("dNBR (Pre - Post)")
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

print(f"dNBR range: [{np.nanmin(dnbr):.3f}, {np.nanmax(dnbr):.3f}]")
print(f"dNBR mean:  {np.nanmean(dnbr):.3f}")

## 3. Burn Severity Classification

Classify dNBR into five severity classes using thresholds from the project config
(following Miller & Thode, 2007).

In [ ]:
severity = classify_severity(dnbr, config)

# Custom colormap for severity classes
sev_cmap = mcolors.ListedColormap(
    ["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"]
)
sev_bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
sev_norm = mcolors.BoundaryNorm(sev_bounds, sev_cmap.N)
sev_labels = ["Unburned", "Low", "Moderate-Low", "Moderate-High", "High"]

fig, ax = plt.subplots(figsize=(8, 6))
display = severity.astype(np.float32)
display[severity == 255] = np.nan

im = ax.imshow(display, cmap=sev_cmap, norm=sev_norm)
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3, 4], shrink=0.8)
cbar.ax.set_yticklabels(sev_labels)
ax.set_title("Burn Severity Classification (dNBR)")
plt.tight_layout()
plt.show()

## 4. Recovery Time Series

Simulate 5 years of post-fire NDVI recovery and fit exponential curves per severity class.

In [ ]:
# Generate synthetic annual NDVI recovery rasters
rng = np.random.default_rng(42)
years_post = config["processing"]["years_post_fire"]
annual_rasters = []

for yr in range(1, years_post + 1):
    base = 0.2 + 0.12 * yr + rng.normal(0, 0.02, pre_nir.shape)
    base = np.where(severity == 0, 0.7 + rng.normal(0, 0.02, pre_nir.shape), base)
    for sev_cls in range(1, 5):
        base[severity == sev_cls] -= sev_cls * 0.05
    annual_rasters.append(np.clip(base, 0, 1).astype(np.float32))

# Build time series and fit model
recovery_ts = build_recovery_timeseries(annual_rasters, severity, config)
recovery_model = fit_recovery_model(recovery_ts, config)

# Plot recovery curves
fig, ax = plt.subplots(figsize=(10, 6))
colors = {0: "#1a9641", 1: "#a6d96a", 2: "#ffffbf", 3: "#fdae61", 4: "#d7191c"}
labels = {0: "Unburned", 1: "Low", 2: "Moderate-Low", 3: "Moderate-High", 4: "High"}

for sev_cls in sorted(recovery_ts["severity_class"].unique()):
    grp = recovery_ts[recovery_ts["severity_class"] == sev_cls]
    ax.plot(
        grp["year"], grp["mean_index"],
        marker="o", color=colors.get(sev_cls, "gray"),
        label=labels.get(sev_cls, f"Class {sev_cls}"),
    )
    ax.fill_between(
        grp["year"],
        grp["mean_index"] - grp["std_index"],
        grp["mean_index"] + grp["std_index"],
        alpha=0.2, color=colors.get(sev_cls, "gray"),
    )

ax.set_xlabel("Years Post-Fire")
ax.set_ylabel("Mean NDVI")
ax.set_title("Vegetation Recovery by Burn Severity Class")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Summary Statistics

In [ ]:
# Severity class distribution
valid = severity[severity != 255]
total = len(valid)
summary_rows = []
for code, label in labels.items():
    count = int(np.count_nonzero(valid == code))
    pct = 100.0 * count / max(total, 1)
    summary_rows.append({"Class": label, "Code": code, "Pixels": count, "Percent": f"{pct:.1f}%"})

summary_df = pd.DataFrame(summary_rows)
print("=== Severity Class Distribution ===")
print(summary_df.to_string(index=False))
print(f"\nTotal valid pixels: {total}")
print(f"Mean dNBR: {np.nanmean(dnbr):.4f}")
print(f"Median dNBR: {np.nanmedian(dnbr):.4f}")

# Recovery model parameters
print("\n=== Exponential Recovery Model Fits ===")
if not recovery_model.empty:
    display_model = recovery_model.copy()
    display_model["severity_label"] = display_model["severity_class"].map(labels)
    print(display_model[
        ["severity_label", "a", "b", "c", "years_to_90pct_recovery", "r_squared"]
    ].to_string(index=False))
else:
    print("No recovery model fitted (insufficient data points).")